#### Online Orders Data Type Conversion

In [36]:
import pandas as pd

## Part A — Diagnose the Raw File

### Load the dataset and inspect head(), dtypes, and info().

In [37]:
df=pd.read_csv("case2_online_orders_raw.csv")
df.head(3)

,Order_ID,Product,Quantity,Unit_Price,Order_Time,Status,Priority,Category,Rating
0,ORD001,Laptop Stand,2,"1,499.50",05-07-2026 10:30,Delivered,Yes,Electronics,4.5
1,ORD002,Wireless Mouse,3,799,05/07/2026 11:15,Delivered,No,Electronics,4
2,ORD003,Office Chair,one,6500,06-07-2026 09:00,Shipped,YES,Furniture,4.2


In [38]:
df.dtypes

Order_ID      str
Product       str
Quantity      str
Unit_Price    str
Order_Time    str
Status        str
Priority      str
Category      str
Rating        str
dtype: object

In [39]:
df.info

<bound method DataFrame.info of    Order_ID          Product Quantity Unit_Price        Order_Time     Status  \
0    ORD001    Laptop Stand         2   1,499.50  05-07-2026 10:30  Delivered   
1    ORD002   Wireless Mouse        3        799  05/07/2026 11:15  Delivered   
2    ORD003     Office Chair      one       6500  06-07-2026 09:00    Shipped   
3    ORD004    Notebook Pack        5       350   06/07/2026 14:45    Pending   
4    ORD005          USB Hub        2      1,250          bad date  Delivered   
5    ORD006        Desk Lamp      NaN       1800  07-07-2026 16:20  Cancelled   
6    ORD007          Pen Set       10        250  08/07/2026 12:00  Delivered   
7    ORD008        Bookshelf        1      8,750  08-07-2026 17:10    Shipped   
8    ORD009           Webcam    2 pcs       3200  09/07/2026 10:05  Delivered   
9    ORD010      Marker Pack        6        300  09-07-2026 13:25    Pending   
10   ORD011           Router        2       2600  10/07/2026 09:40  Delivered

### List the columns that should be numeric, datetime, Boolean, or category.

Numberic = Quantity , unit price , rating <br/>
Datetime = order_time<br/>
Boolean = Priority<br/>
Category = category

### Find examples of values that will fail if converted directly.

ORD001, Laptop Stand ,2,"1,499.50",05-07-2026 10:30,Delivered,Yes,Electronics,4.5 <br>
ORD007,Pen Set,10,250,08/07/2026 12:00,Delivered,FALSE,Stationery,4.1

## Part B — Product and Numeric Fields

#### Create df_clean as a copy and remove leading/trailing spaces from Product.

In [40]:
df_clean=df.copy()
df_clean["Product"] = df_clean["Product"].str.strip()
df_clean.head()

,Order_ID,Product,Quantity,Unit_Price,Order_Time,Status,Priority,Category,Rating
0,ORD001,Laptop Stand,2,"1,499.50",05-07-2026 10:30,Delivered,Yes,Electronics,4.5
1,ORD002,Wireless Mouse,3,799,05/07/2026 11:15,Delivered,No,Electronics,4
2,ORD003,Office Chair,one,6500,06-07-2026 09:00,Shipped,YES,Furniture,4.2
3,ORD004,Notebook Pack,5,350,06/07/2026 14:45,Pending,N,Stationery,Not Rated
4,ORD005,USB Hub,2,"1,250",bad date,Delivered,Y,Electronics,4.7


#### Convert Quantity to nullable Int64 using safe numeric conversion.

In [41]:
df_clean["Quantity"] = pd.to_numeric(df_clean["Quantity"],errors="coerce").astype("Int64")
df_clean.dtypes

Order_ID        str
Product         str
Quantity      Int64
Unit_Price      str
Order_Time      str
Status          str
Priority        str
Category        str
Rating          str
dtype: object

#### Identify the Quantity values that became missing after conversion.

In [42]:
print(df[df_clean["Quantity"].isna()])

  Order_ID       Product Quantity Unit_Price        Order_Time     Status  \
2   ORD003  Office Chair      one       6500  06-07-2026 09:00    Shipped   
5   ORD006     Desk Lamp      NaN       1800  07-07-2026 16:20  Cancelled   
8   ORD009        Webcam    2 pcs       3200  09/07/2026 10:05  Delivered   

  Priority     Category Rating  
2      YES    Furniture    4.2  
5       No    Furniture    3.9  
8      Yes  Electronics    4.8  


#### Clean commas and spaces from Unit_Price and convert it to numeric.

In [47]:
df_clean["Unit_Price"] = pd.to_numeric(df_clean["Unit_Price"].astype(str).str.replace(",", "", regex=False).str.replace(" ", "", regex=False))
print(df_clean.dtypes,df_clean["Unit_Price"])

Order_ID          str
Product           str
Quantity        Int64
Unit_Price    float64
Order_Time        str
Status            str
Priority          str
Category          str
Rating            str
dtype: object 0     1499.5
1      799.0
2     6500.0
3      350.0
4     1250.0
5     1800.0
6      250.0
7     8750.0
8     3200.0
9      300.0
10    2600.0
11    9500.0
Name: Unit_Price, dtype: float64


#### Explain why '1,499.50' cannot be handled correctly by a simple astype(float) before cleaning.

Because "1,499.50" is a formatted string, not a valid Python floating-point literal. The comma must first be removed:

## Part C — Date/Time and Rating

#### Convert Order_Time to datetime while allowing mixed day-first formats.

In [51]:
df_clean["Order_Time"] = pd.to_datetime(df_clean["Order_Time"],dayfirst=True,format="mixed",errors='coerce')
df_clean.dtypes

Order_ID                 str
Product                  str
Quantity               Int64
Unit_Price           float64
Order_Time    datetime64[us]
Status                   str
Priority                 str
Category                 str
Rating                   str
dtype: object

#### Identify the invalid Order_Time that becomes NaT.

In [54]:
df_clean[df_clean["Order_Time"].isna()]

,Order_ID,Product,Quantity,Unit_Price,Order_Time,Status,Priority,Category,Rating
1,ORD002,Wireless Mouse,3,799.0,NaT,Delivered,No,Electronics,4
3,ORD004,Notebook Pack,5,350.0,NaT,Pending,N,Stationery,Not Rated
4,ORD005,USB Hub,2,1250.0,NaT,Delivered,Y,Electronics,4.7
6,ORD007,Pen Set,10,250.0,NaT,Delivered,FALSE,Stationery,4.1
8,ORD009,Webcam,<NA>,3200.0,NaT,Delivered,Yes,Electronics,4.8
10,ORD011,Router,2,2600.0,NaT,Delivered,Y,Electronics,4.3


#### Convert Rating to numeric using errors='coerce'.

In [55]:
df_clean["Rating"] = pd.to_numeric(df_clean["Rating"],errors='coerce')
df_clean.dtypes


Order_ID                 str
Product                  str
Quantity               Int64
Unit_Price           float64
Order_Time    datetime64[us]
Status                   str
Priority                 str
Category                 str
Rating               float64
dtype: object

#### Identify the rating value that becomes NaN.

In [56]:
df_clean[df_clean["Rating"].isna()]

,Order_ID,Product,Quantity,Unit_Price,Order_Time,Status,Priority,Category,Rating
3,ORD004,Notebook Pack,5,350.0,NaT,Pending,N,Stationery,NaN


## Part D — Boolean and Category

#### Standardize Priority values such as Yes, Y, TRUE, No, N, and FALSE and convert them to nullable boolean

In [61]:
priority_map = {
    "yes": True,
    "y": True,
    "true": True,
    "no": False,
    "n": False,
    "false": False
}

df_clean["Priority"] = df["Priority"].astype(str).str.strip().str.lower().map(priority_map).astype(bool)
df_clean.head(2)

,Order_ID,Product,Quantity,Unit_Price,Order_Time,Status,Priority,Category,Rating
0,ORD001,Laptop Stand,2,1499.5,2026-07-05 10:30:00,Delivered,True,Electronics,4.5
1,ORD002,Wireless Mouse,3,799.0,NaT,Delivered,False,Electronics,4.0


#### Convert Status to category.

In [62]:
df_clean["Status"] = df_clean["Status"].astype("category")
df_clean["Status"]

0     Delivered
1     Delivered
2       Shipped
3       Pending
4     Delivered
5     Cancelled
6     Delivered
7       Shipped
8     Delivered
9       Pending
10    Delivered
11      Shipped
Name: Status, dtype: category
Categories (4, str): ['Cancelled', 'Delivered', 'Pending', 'Shipped']

#### Convert Category to category.

In [63]:
df_clean["Category"] = df_clean["Category"].astype("category")
df_clean["Category"]

0     Electronics
1     Electronics
2       Furniture
3      Stationery
4     Electronics
5       Furniture
6      Stationery
7       Furniture
8     Electronics
9      Stationery
10    Electronics
11      Furniture
Name: Category, dtype: category
Categories (3, str): ['Electronics', 'Furniture', 'Stationery']

#### Display the category labels for Status and Category.

In [64]:
print(df_clean["Status"].cat.categories)
print(df_clean["Category"].cat.categories)

Index(['Cancelled', 'Delivered', 'Pending', 'Shipped'], dtype='str')
Index(['Electronics', 'Furniture', 'Stationery'], dtype='str')


## Part E — Final Verification

#### Display all final dtypes.

In [65]:
df_clean.dtypes

Order_ID                 str
Product                  str
Quantity               Int64
Unit_Price           float64
Order_Time    datetime64[us]
Status              category
Priority                bool
Category            category
Rating               float64
dtype: object

#### Count conversion-created missing values in Quantity, Order_Time, and Rating.

In [66]:
print("Quantity:", df_clean["Quantity"].isna().sum())
print("Order_Time:", df_clean["Order_Time"].isna().sum())
print("Rating:", df_clean["Rating"].isna().sum())

Quantity: 3
Order_Time: 6
Rating: 1


#### Display rows where at least one of those three fields is missing.

In [67]:
missing_rows = df_clean[df_clean[["Quantity", "Order_Time", "Rating"]].isna().any(axis=1)]
missing_rows

,Order_ID,Product,Quantity,Unit_Price,Order_Time,Status,Priority,Category,Rating
1,ORD002,Wireless Mouse,3,799.0,NaT,Delivered,False,Electronics,4.0
2,ORD003,Office Chair,<NA>,6500.0,2026-07-06 09:00:00,Shipped,True,Furniture,4.2
3,ORD004,Notebook Pack,5,350.0,NaT,Pending,False,Stationery,NaN
4,ORD005,USB Hub,2,1250.0,NaT,Delivered,True,Electronics,4.7
5,ORD006,Desk Lamp,<NA>,1800.0,2026-07-07 16:20:00,Cancelled,False,Furniture,3.9
6,ORD007,Pen Set,10,250.0,NaT,Delivered,False,Stationery,4.1
8,ORD009,Webcam,<NA>,3200.0,NaT,Delivered,True,Electronics,4.8
10,ORD011,Router,2,2600.0,NaT,Delivered,True,Electronics,4.3


#### Explain the difference between NaN, NaT, and <NA> in the final result.

NaN = missing value generally used with floating-point/numeric data.<br>
NaT = "Not a Time"; missing/invalid datetime value.<br>
NA = Pandas' nullable missing-value marker, used here by nullable Int64.

#### Save your converted dataset and compare it with the supplied clean dataset.

In [68]:
df_clean.to_csv("case2_online_orders_clean.csv",index=False)

## Part F — Short Interpretation Questions

#### Why is errors='coerce' safer than forcing a conversion on messy imported data?

errors='coerce' prevents the entire conversion from failing when messy data is encountered. Invalid values are converted to missing values, allowing the remaining valid data to be processed.

#### Why should commas be removed from price strings before numeric conversion?


Commas are formatting characters in numbers such as:<br>
1,499.50<br>
They prevent direct numeric conversion. Removing them produces:<br>
1499.50<br>
which Pandas can convert to a numeric value.

#### Why are Status and Category good candidates for category dtype?


Using category is appropriate for such repeated categorical values and can reduce memory usage.

#### Why is Order_ID better kept as text rather than converted to category or number?

They are labels/identifiers, not quantities. Therefore, converting them to numbers would be inappropriate. Keeping them as text preserves the original identifier exactly.